# Monte Carlo methods: learning values from full episodes

_Reinforcement Learning_

**No model of the world? Play whole episodes to the end and average the returns you actually got.**

So far in this curriculum, value iteration (ai-value-iteration) computed values by
       knowing the model: it used the transition probabilities $P(s'\mid s,a)$ and rewards directly in
       the Bellman equation. But in most real problems you do not have those equations. You can only
       act and observe.

       Monte Carlo (MC) methods &mdash; named after the casino, because they learn from random
       sampling &mdash; are the first model-free idea: estimate values directly from experience,
       from sampled complete episodes, with zero knowledge of $P$ or $R$.

---

This notebook is a practice scaffold for the **Monte Carlo methods: learning values from full episodes** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + numpy

In [ ]:
# First-visit Monte Carlo PREDICTION (model-free policy evaluation).
# Colab: !pip install gymnasium
import numpy as np
import gymnasium as gym
from collections import defaultdict

env = gym.make("Blackjack-v1", sab=True)   # sab=True -> classic Sutton & Barto rules
gamma = 1.0                                 # Blackjack is undiscounted (episodes are short)

# A fixed policy to EVALUATE: stick (action 0) on 20 or 21, else hit (action 1).
def policy(state):
    player_sum, dealer_card, usable_ace = state
    return 0 if player_sum >= 20 else 1

V = defaultdict(float)                      # value estimate V(s)
N = defaultdict(int)                        # how many returns averaged into V(s)

def run_episode():
    """Play one hand; return the list of (state, reward) actually seen."""
    state, _ = env.reset()
    trajectory = []
    done = False
    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        trajectory.append((state, reward))
        state = next_state
        done = terminated or truncated
    return trajectory

n_episodes = 500_000
for _ in range(n_episodes):
    traj = run_episode()
    visited = set()                         # FIRST-VISIT: count each state once per episode
    G = 0.0
    # Walk the episode BACKWARDS so the return accumulates in one pass:
    # G_t = r_{t+1} + gamma * G_{t+1}.
    for state, reward in reversed(traj):
        G = reward + gamma * G
        if state not in visited:            # first occurrence of this state in the episode
            visited.add(state)
            N[state] += 1
            V[state] += (G - V[state]) / N[state]   # incremental mean: V <- V + (G - V)/N

# Inspect a few states (player_sum, dealer_showing, usable_ace).
for s in [(21, 10, False), (20, 7, False), (18, 6, False), (13, 10, False), (12, 4, True)]:
    print(f"V{ s } = {V[s]:+.3f}   (averaged over {N[s]} first-visits)")

# Typical output (values are expected return = P(win) - P(loss), in [-1, +1]):
# V(21, 10, False) = +0.892   (averaged over ~13000 first-visits)
# V(20,  7, False) = +0.773   (averaged over ~10000 first-visits)
# V(18,  6, False) = -0.100   (averaged over ~ 8000 first-visits)
# V(13, 10, False) = -0.535   (averaged over ~ 9000 first-visits)
# V(12,  4, True ) = -0.090   (averaged over ~ 1200 first-visits)
# Sticking only on 20/21 is great when you already hold 20-21, poor otherwise --
# exactly what an UNBIASED average of real returns reveals, with no model of the deck.

## Visualize the data & results

_As Monte Carlo accumulates episodes, does its estimate of a state's value home in on the true value -- and what does 'unbiased but high-variance' look like on the way there?_

In [ ]:
import numpy as np

# Tiny self-contained episodic MDP: a 3-state chain s0 -> s1 -> s2 -> TERMINAL.
# Transitions are deterministic, but the TERMINAL reward is noisy, so the return
# G from s0 has real variance -- the whole point of the demo. gamma = 0.9.
gamma = 0.9
rng = np.random.default_rng(0)

def episode_return_from_s0():
    # rewards: -1 (s0->s1), -1 (s1->s2), and -1 + Normal(10, 4) at the terminal step.
    rewards = [-1.0, -1.0, -1.0 + rng.normal(10.0, 4.0)]
    G = 0.0
    for r in reversed(rewards):          # G_t = r_{t+1} + gamma * G_{t+1}
        G = r + gamma * G
    return G

# True value (the noisy bonus averages to +10, minus the -1 terminal step -> +9):
true_V = -1 + gamma * (-1) + gamma**2 * (-1 + 10.0)
print("true V(s0) =", round(true_V, 4))            # -> 5.39

# First-visit MC prediction: running mean of returns (incremental update).
N_EP = 2000
V, N = 0.0, 0
running = []
for _ in range(N_EP):
    G = episode_return_from_s0()
    N += 1
    V += (G - V) / N                                # V <- V + (G - V)/N
    running.append(V)

# Subsample to <= 60 points (log-ish spacing) for plotting.
idxs = [1,2,3,4,5,7,10,15,20,30,40,50,70,100,150,200,300,400,500,700,1000,1300,1600,2000]
for i in idxs:
    print(f"episodes={i:>4}  V_hat(s0) = {running[i-1]:.3f}")
# episodes=   1  V_hat(s0) = 5.797
# episodes=   5  V_hat(s0) = 5.522
# episodes=  15  V_hat(s0) = 4.629   <- high-variance early swing
# episodes= 100  V_hat(s0) = 5.653
# episodes= 500  V_hat(s0) = 5.303
# episodes=2000  V_hat(s0) = 5.299   -> settling toward true 5.39

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Three episodes following policy $\pi$ pass through state $s$ and yield returns $G = 4$, $G = 10$, $G = 7$. Give the first-visit MC estimate of $V^\pi(s)$, then redo it with the incremental-mean update to confirm it matches.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Batch mean: add the returns and divide by how many. — _MC estimates $V^\pi(s)=\mathbb{E}_\pi[G\mid s]$ by the sample mean of observed returns._
- $V(s) = (4 + 10 + 7)/3 = 21/3 = 7$. — _That is the average of the three returns._
- Incremental: start $V=0,N=0$. $V\leftarrow 0+\tfrac11(4-0)=4$; $V\leftarrow 4+\tfrac12(10-4)=7$; $V\leftarrow 7+\tfrac13(7-7)=7$. — _The update $V\leftarrow V+\tfrac1N(G-V)$ provably equals the running mean._

**Answer:** $V^\pi(s) = 7$. Both the batch mean and the incremental update give exactly $7$ &mdash; they are the same computation.

</details>

**Problem 2.** Why does MC prediction need the task to be episodic (every run must reach a terminal state)? Name one fix when the task never ends.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what MC averages: the return $G_t=\sum_{k\ge t}\gamma^{\,k-t}r_k$. — _The update only happens once you can compute $G_t$._
- If the episode never terminates, that sum has no final term, so $G_t$ is never available to record. — _MC must wait for the end of the episode; with no end, there is no complete return to average._

**Answer:** MC averages complete returns, and a complete return only exists if the episode ends. On a continuing task you can impose an artificial horizon (cut episodes off at a fixed length) or switch to a bootstrapping method like TD (aix-sarsa-td), which updates each step without waiting for termination.

</details>

**Problem 3.** In MC control, why do we estimate the action-value $Q(s,a)$ instead of the state-value $V(s)$, given that the model is unknown?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the greedy improvement on $V$: $\pi'(s)=\arg\max_a \sum_{s'} P(s'\mid s,a)\,[\,r + \gamma V(s')\,]$. — _Being greedy on $V$ requires the transition probabilities $P$ to look one step ahead._
- Model-free MC has no $P$, so it cannot evaluate that $\arg\max$. — _That is exactly the information MC refuses to assume._
- Write the greedy improvement on $Q$: $\pi'(s)=\arg\max_a Q(s,a)$. — _$Q$ already folds in the one-step lookahead, so choosing the best action needs no model._

**Answer:** Greedy-on-$V$ needs the model $P$ to compare actions; greedy-on-$Q$ does not, because $Q(s,a)$ already encodes "take $a$, then follow $\pi$". With the model unknown, MC control therefore estimates $Q$ and improves via $\pi'(s)=\arg\max_a Q(s,a)$, keeping exploration with exploring starts or an $\epsilon$-greedy policy (GLIE: Greedy in the Limit with Infinite Exploration).

</details>